In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")          # ca notebook-ul sa gaseasca folderul src/
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import evaluare as ev

In [2]:
#verificare
df = ev.incarca_date("../data/processed/cipro_norvegia.tsv")
train_t, test_t = ev.split_temporal(df, 2012)
print(df.shape, len(train_t), len(test_t))
ev.evalueaza(train_t, test_t).round(3)

(3148, 14) 1723 1425


,AUROC,AUPRC_castig
ST,0.877,0.445
HC0,0.500,0.000
HC2,0.500,0.000
HC5,0.502,0.000
HC10,0.526,0.041
HC20,0.665,0.218
HC50,0.857,0.509
HC100,0.897,0.573


In [3]:
#impartirea pe clustere
frac = len(test_t) / len(df)
niveluri_split = ["aleatoriu", 10, 20, 50, 100]
modele_arata = ["ST", "HC20", "HC50", "HC100"]
rez = {}
for nivel in niveluri_split:
    scoruri = {m: [] for m in modele_arata}
    for seed in range(20):
        if nivel == "aleatoriu":
            tr, te = ev.split_aleatoriu(df, len(df) - len(test_t), len(test_t), seed)
        else:
            tr, te = ev.split_pe_clustere(df, nivel, frac, seed)
        if te["y"].nunique() < 2 or tr["y"].nunique() < 2:
            continue
        for m in modele_arata:
            scor = ev.model_linie(tr, te, ev.MODELE[m])
            scoruri[m].append(ev.metrici(te["y"].values, scor)["AUROC"])
    rez[nivel] = {m: np.nanmean(v) for m, v in scoruri.items()}
tab_cl = pd.DataFrame(rez).round(3)
tab_cl.columns = ["aleatoriu" if c == "aleatoriu" else f"HC{c}" for c in tab_cl.columns]
tab_cl

,aleatoriu,HC10,HC20,HC50,HC100
ST,0.893,0.889,0.877,0.760,0.703
HC20,0.765,0.722,0.500,0.500,0.500
HC50,0.895,0.886,0.860,0.500,0.500
HC100,0.913,0.914,0.893,0.732,0.500


In [4]:
#fractia reala de test
for nivel in [10, 20, 50, 100]:
    tr, te = ev.split_pe_clustere(df, nivel, frac, 0)
    print(f"split la HC{nivel}: test = {len(te)} izolate ({len(te)/len(df):.2f}), prevalenta test = {te['y'].mean():.3f}")

split la HC10: test = 1425 izolate (0.45), prevalenta test = 0.095
split la HC20: test = 1425 izolate (0.45), prevalenta test = 0.074
split la HC50: test = 1425 izolate (0.45), prevalenta test = 0.072
split la HC100: test = 1743 izolate (0.55), prevalenta test = 0.079


In [5]:
#controlul cu distanta potrivita
rez_c = [ev.control_distanta(df, seed=s) for s in range(300)]
rez_c = pd.DataFrame([r for r in rez_c if r is not None])
print(len(rez_c), "experimente valide din 300")
print(rez_c[["n_antrenare", "n_control", "prev_temporal", "prev_control"]].mean().round(3))
coloane = ["AUROC_temporal", "AUROC_control", "delta_AUROC", "castig_temporal", "castig_control", "delta_castig"]
rezumat = pd.DataFrame({"medie": rez_c[coloane].mean(),
                        "IC95_jos": rez_c[coloane].quantile(0.025),
                        "IC95_sus": rez_c[coloane].quantile(0.975)}).round(3)
rezumat

300 experimente valide din 300
n_antrenare      1119.000
n_control         399.527
prev_temporal       0.137
prev_control        0.062
dtype: float64


,medie,IC95_jos,IC95_sus
AUROC_temporal,0.843,0.805,0.878
AUROC_control,0.867,0.788,0.940
delta_AUROC,-0.024,-0.104,0.065
castig_temporal,0.480,0.387,0.548
castig_control,0.452,0.206,0.700
delta_castig,0.027,-0.227,0.301
